In [3]:
!pip show transformers torch torchvision protobuf qwen-vl-utils

Name: transformers
Version: 5.17.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /opt/homebrew/lib/python3.11/site-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: ane_transformers, compressed-tensors, flash-linear-attention, metal_liger, mlx-audio, mlx-lm, mlx-vlm, optimum, optimum-onnx, peft, sentence-transformers, trl, vllm, xgrammar
---
Name: torch
Version: 2.14.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: P

In [14]:
!git clone https://github.com/mpsops/mps-flash-attention

Cloning into 'mps-flash-attention'...
remote: Enumerating objects: 507, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 507 (delta 107), reused 135 (delta 88), pack-reused 346 (from 1)
Receiving objects: 100% (507/507), 2.69 MiB | 3.27 MiB/s, done.
Resolving deltas: 100% (255/255), done.


## transformers float 32

In [1]:

import time
import torch
import transformers
from transformers import AutoConfig, AutoProcessor, Qwen3VLForConditionalGeneration
from PIL import Image

BASE_MODEL_ID = "Qwen3VL-2B-Medmaxv11"


model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    dtype=torch.float32,
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID, 
    trust_remote_code=True,
)
model.eval()


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 626/626 [00:00<00:00, 675.28it/s]


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [6]:

prompt = """Write an essay on topic of 'Epigenetics and Intergenerational Trauma: Molecular Inheritance of Societal Stress' in 5 paragrpahs.
Output Format:

paragraph1: <long explanation>

paragraph2: <long explanation>

paragraph3: <long explanation>

paragraph4: <long explanation>

paragraph5: <long explanation>
"""


messages = [
    {
        "role": "system",
        "content": "You are an expert medical educator."
    },
    {
        "role": "user",
        "content": [
            #{"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }
]


with torch.inference_mode():
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=text,
        return_tensors="pt",
    ).to("mps")

    # --- Start Benchmarking ---
    torch.mps.synchronize()  # Wait for all previous GPU tasks to finish
    start_time = time.perf_counter()

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=4096,    
        do_sample=False,
 
    )
    
    torch.mps.synchronize() 
    end_time = time.perf_counter()

    # Isolate only the newly generated tokens
    new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]

    output_text = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0]

    # Calculate Speed Metrics
    num_generated_tokens = new_tokens.shape[1]
    generation_time = end_time - start_time
    tokens_per_second = num_generated_tokens / generation_time


In [7]:

# Print Results
print("=== Output ===")
print(output_text.strip())
print("\n=== Performance Metrics ===")
print(f"Total New Tokens:  {num_generated_tokens}")
print(f"Generation Time:   {generation_time:.4f} seconds")
print(f"Token Speed:       {tokens_per_second:.2f} tokens/sec")

=== Output ===
</think>

paragraph1: Epigenetics refers to the study of changes in gene expression that occur without altering the DNA sequence. These changes can be influenced by environmental factors and can be passed down through generations. In the context of intergenerational trauma, epigenetic mechanisms might play a role in transmitting stress responses from one generation to the next. This could involve changes in how genes are regulated, potentially affecting stress response pathways and resilience mechanisms.

paragraph2: Intergenerational trauma refers to the transmission of trauma and its effects across generations. This can occur through various mechanisms, including psychological, biological, and social factors. Epigenetic changes could be one of these mechanisms, where trauma experienced by one generation might alter the epigenetic markers in offspring, influencing their stress response and potentially affecting their ability to cope with stress.

paragraph3: Epigenetic 

## transformers bfloat16

In [1]:
import os
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"

import time
import torch
import transformers
from transformers import AutoConfig, AutoProcessor, Qwen3VLForConditionalGeneration
from PIL import Image

BASE_MODEL_ID = "Qwen3VL-2B-Medmaxv11"


model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID, 
    trust_remote_code=True,
)
model.eval()


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 626/626 [00:02<00:00, 253.73it/s]


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [7]:
import time

prompt = """Write an essay on topic of 'Epigenetics and Intergenerational Trauma: Molecular Inheritance of Societal Stress' in 5 paragrpahs.
Output Format:

paragraph1: <long explanation>

paragraph2: <long explanation>

paragraph3: <long explanation>

paragraph4: <long explanation>

paragraph5: <long explanation>
"""


messages = [
    {
        "role": "system",
        "content": "You are an expert medical educator."
    },
    {
        "role": "user",
        "content": [
            #{"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }
]


with torch.inference_mode():
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=text,
        #images=[img],
        return_tensors="pt",
    ).to("mps")

    # --- Start Benchmarking ---
    torch.mps.synchronize()  # Wait for all previous GPU tasks to finish
    start_time = time.perf_counter()

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=4096,      # tight control (prevents drift)
        do_sample=False,        # deterministic output
 
    )
    
    torch.mps.synchronize()  # Wait for generation to completely finish
    end_time = time.perf_counter()
    # --- End Benchmarking ---

    # Isolate only the newly generated tokens
    new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]

    output_text = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0]

    # Calculate Speed Metrics
    num_generated_tokens = new_tokens.shape[1]
    generation_time = end_time - start_time
    tokens_per_second = num_generated_tokens / generation_time


In [8]:

# Print Results
print("=== Output ===")
print(output_text.strip())
print("\n=== Performance Metrics ===")
print(f"Total New Tokens:  {num_generated_tokens}")
print(f"Generation Time:   {generation_time:.4f} seconds")
print(f"Token Speed:       {tokens_per_second:.2f} tokens/sec")

=== Output ===
</think>

paragraph1: Epigenetics refers to the study of changes in gene expression that do not involve changes to the DNA sequence itself. These changes can be influenced by environmental factors and can be passed down through generations. In the context of intergenerational trauma, epigenetic mechanisms might play a role in transmitting stress responses from one generation to the next. This could involve changes in how genes are regulated, potentially affecting stress response pathways and resilience mechanisms.

paragraph2: The concept of molecular inheritance of societal stress suggests that societal stressors, such as trauma or adversity, can be transmitted through epigenetic changes. These changes might alter the expression of stress-related genes, making individuals more susceptible to stress or trauma in subsequent generations. This could be particularly relevant in understanding how historical trauma, such as that experienced by certain communities, might affect

## metalliger float 32

In [19]:
!ls

mps-flash-attention


In [1]:
from PIL import Image
import torch
from metalliger.inference import InferenceEngine

engine = InferenceEngine(
    model_id="Qwen3VL-2B-Medmaxv11",
    max_seq_len=4096,
    dtype=torch.float32,       # ← Half the memory, what model was fine-tuned in
    use_compile=False
)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Initializing M4 Pro Engine for Qwen3VL-2B-Medmaxv11 (dtype=torch.float32)...


Loading weights: 100%|████████████████████| 626/626 [00:00<00:00, 7461.97it/s]
MetalLiger Flash Attention: mps_flash_attn reports not available (C++ extension may not be built). Skipping.


- Injecting MetalLiger Fused Kernels (RMSNorm, SwiGLU)...
- Flash Attention: active (prefill speedup)
   RMSNorm fast-path: 57 layers optimized (fused F.rms_norm)
   SwiGLU fast-path: 28 MLP layers optimized for inference (torch.float32)
- Running in stable eager mode with kernel fusions.
 Engine ready in 7.30s


In [13]:


prompt = """Write an essay on topic of 'Epigenetics and Intergenerational Trauma: Molecular Inheritance of Societal Stress' in 5 paragrpahs.
Output Format:

paragraph1: <long explanation>

paragraph2: <long explanation>

paragraph3: <long explanation>

paragraph4: <long explanation>

paragraph5: <long explanation>
"""


messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt}
        ]
    }
]

# 5. Generate
response = engine.generate(
    messages=messages,
    max_new_tokens=1048,
    temperature=0.0,
)
print("\n--- Output ---")
print(response)

NameError: name 'engine' is not defined

## metalliger bfloat 16

In [1]:
import torch
from metalliger.inference import InferenceEngine

engine = InferenceEngine(
    model_id="Qwen3VL-2B-Medmaxv11",
    max_seq_len=4096,
    dtype=torch.bfloat16,       # ← Half the memory, what model was fine-tuned in
    use_compile=True
)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Initializing M4 Pro Engine for Qwen3VL-2B-Medmaxv11 (dtype=torch.bfloat16, memory_fraction=1.0)...


Loading weights: 100%|█████████████████████| 626/626 [00:04<00:00, 142.08it/s]


⚡ Injecting MetalLiger Fused Kernels (RMSNorm, SwiGLU)...
   RMSNorm fast-path: 57 layers optimized (fused F.rms_norm)
   SwiGLU fast-path: 28 MLP layers optimized for inference (torch.bfloat16)
🔥 Compiling transformer layers individually (backend=inductor, dynamic=True)...
   Compiled 24 layers with inductor.
🔥 Warming up compiled graphs...
   Warmup complete.
✅ Engine ready in 7.02s


In [2]:


prompt = """Write an essay on topic of 'Epigenetics and Intergenerational Trauma: Molecular Inheritance of Societal Stress' in 5 paragrpahs.
Output Format:

paragraph1: <long explanation>

paragraph2: <long explanation>

paragraph3: <long explanation>

paragraph4: <long explanation>

paragraph5: <long explanation>
"""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt}
        ]
    }
]

# 5. Generate
response = engine.generate(
    messages=messages,
    max_new_tokens=1048,
    temperature=0.0,
)

print("\n--- Output ---")
print(response)

🧠 Prefilling prompt...
   Prefill done in 0.085s (979.6 tokens/s, 83 tokens)
⚡ Decoding...
   Decode done in 7.624s (48.7 tokens/s, 371 tokens)

--- Output ---
</think>

paragraph1: Epigenetics, the study of changes in gene expression that do not involve changes to the DNA sequence, plays a crucial role in understanding how societal stress can be passed down through generations. Epigenetic mechanisms, such as DNA methylation and histone modification, can influence gene expression in response to environmental factors, including stress. These changes can be inherited by offspring, potentially affecting their health and behavior. This molecular inheritance of societal stress highlights the importance of considering epigenetic factors in understanding intergenerational trauma and its impacts.

paragraph2: The concept of epigenetic inheritance suggests that societal stress experienced by parents can be transmitted to their children through epigenetic modifications. These modifications can a

## mlx-vlm bloat16

In [3]:
import time
from mlx_vlm import load, generate

# 1. Point this to the local folder where you fixed the config.json
MODEL_PATH = "./Madrimed_vl_2b_mlx"

print("Loading model into Apple Unified Memory...")
model, processor = load(MODEL_PATH)


Loading model into Apple Unified Memory...


In [6]:

prompt =  """Write an essay on topic of 'Epigenetics and Intergenerational Trauma: Molecular Inheritance of Societal Stress' in 5 paragrpahs.
Output Format:

paragraph1: <long explanation>

paragraph2: <long explanation>

paragraph3: <long explanation>

paragraph4: <long explanation>

paragraph5: <long explanation>
"""

# --- THE FIX: Format the prompt using the Chat Template ---
messages = [
    {
        "role": "system",
        "content": "You are an expert medical AI. You must deeply analyze the question and provide the final answer."
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt}
        ]
    }
]

# Apply the template so Qwen knows exactly where to put the 65 image features
prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\nStarting generation...")
start_time = time.perf_counter()

# 2. MLX generate function handles the Metal optimization natively
output = generate(
    model, 
    processor, 
    prompt,       # Pass the formatted prompt here!
    max_tokens=2048,
    verbose=True # Set to True if you want MLX to print its own exact speed stats
)

end_time = time.perf_counter()


Starting generation...
Files: [] 

Prompt: <|im_start|>system
You are an expert medical AI. You must deeply analyze the question and provide the final answer.<|im_end|>
<|im_start|>user
Write an essay on topic of 'Epigenetics and Intergenerational Trauma: Molecular Inheritance of Societal Stress' in 5 paragrpahs.
Output Format:

paragraph1: <long explanation>

paragraph2: <long explanation>

paragraph3: <long explanation>

paragraph4: <long explanation>

paragraph5: <long explanation>
<|im_end|>
<|im_start|>assistant
<think>



Prefill:  99%|███████████████████████████▋| 106/107 [00:00<00:00, 985.57tok/s]


</think>

paragraph1: Epigenetics refers to the study of changes in gene expression that do not involve changes to the DNA sequence itself. These changes can be influenced by environmental factors, including stress. Intergenerational trauma, which involves the transmission of trauma from one generation to the next, can be influenced by epigenetic mechanisms. For example, stress experienced by parents can lead to epigenetic modifications in their children, affecting their stress responses and potentially leading to intergenerational trauma.

paragraph2: Epigenetic modifications can be passed down through the generations, influencing the stress responses of descendants. This can occur through mechanisms such as DNA methylation, histone modification, and non-coding RNA regulation. These epigenetic changes can alter gene expression patterns, potentially leading to increased vulnerability to stress and trauma.

paragraph3: The molecular inheritance of societal stress suggests that stress ex

## llama float16

In [1]:
!chmod +x ./llama.cpp/build/bin/llama-mtmd-cli

In [ ]:
!./llama.cpp/build/bin/llama-cli \
    -m madrimed1.2_quant/medmax-2b-f16.gguf \
    --mmproj madrimed1.2_quant/mmproj-f16.gguf \
    -p "Write an essay on topic of 'Epigenetics and Intergenerational Trauma: Molecular Inheritance of Societal Stress' in 5 paragraphs.\nOutput Format:\n\nparagraph1: <long explanation>\n\nparagraph2: <long explanation>\n\nparagraph3: <long explanation>\n\nparagraph4: <long explanation>\n\nparagraph5: <long explanation>" \
    #--image-min-tokens 4096 \
    -ngl 99



Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11217-c9064dded
model      : madrimed1.2_quant/medmax-2b-f16.gguf
ftype      : F16
modalities : text, vision, video

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern
  /image <file>       add an image file
  /video <file>       add a video file



> Write an essay on topic of 'Epigenetics and Intergenerational Trauma: Molecular Inheritance of Societal Stress' in 5 paragraphs.
Output Format:

paragraph1: <long explanation>

paragraph2: <long explanation>

paragraph3: <long explanation>

paragraph4: <long explanati

## vq metalliger float 32

In [1]:
from PIL import Image
import torch
from metalliger.inference import InferenceEngine

engine = InferenceEngine(
    model_id="Qwen3VL-2B-Medmaxv11",
    max_seq_len=4096,
    dtype=torch.float32,       # ← Half the memory, what model was fine-tuned in
    use_compile=False
)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Initializing M4 Pro Engine for Qwen3VL-2B-Medmaxv11 (dtype=torch.float32)...


Loading weights: 100%|████████████████████| 626/626 [00:00<00:00, 7319.23it/s]
MetalLiger Flash Attention: mps_flash_attn reports not available (C++ extension may not be built). Skipping.


- Injecting MetalLiger Fused Kernels (RMSNorm, SwiGLU)...
- Flash Attention: active (prefill speedup)
   RMSNorm fast-path: 57 layers optimized (fused F.rms_norm)
   SwiGLU fast-path: 28 MLP layers optimized for inference (torch.float32)
- Running in stable eager mode with kernel fusions.
 Engine ready in 7.40s


In [2]:


def load_direct_image(path: str, max_edge: int = 512) -> Image.Image:
    with Image.open(path) as raw:
        img = raw.convert("RGB")
        width, height = img.size
        if max(width, height) > max_edge:
            scale = max_edge / max(width, height)
            new_w = int(width * scale)
            new_h = int(height * scale)
            img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        import io
        import base64
        buffered = io.BytesIO()
        img.save(buffered, format="JPEG")
        return base64.b64encode(buffered.getvalue()).decode("utf-8")#img


# 3. Prompt & Image
prompt = """You are a expert medical AI given a Histopathology Case (Glioblastoma / Pseudopalisading Necrosis)


1. Detection of central pseudopalisading necrosis (serpiginous necrotic core flanked by crowded hyperchromatic tumor nuclei).
2. Recognition of hyperplastic glomeruloid microvascular proliferation.
3. Alignment with WHO CNS Grade 4 / Glioblastoma and associated $O^6$-alkylating chemotherapy sensitivity (MGMT promoter hypermethylation).
"""

image_path = "GBM_pseudopalisading_necrosis.jpg"
img = load_direct_image(image_path)
# 4. Messages format
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }
]

# 5. Generate
response = engine.generate(
    messages=messages,
    images=[img],
    max_new_tokens=1048,
    temperature=0.0,
    verbose=True
)
print("\n--- Output ---")
print(response)

Prefilling prompt...
   Prefill done in 0.594s (525.0 tokens/s, 312 tokens)
⚡ Decoding...
   Decode done in 5.105s (29.6 tokens/s, 151 tokens)

--- Output ---
</think>

Based on the provided histopathological features, the diagnosis of glioblastoma is confirmed. The presence of central pseudopalisading necrosis, characterized by a serpiginous necrotic core surrounded by crowded hyperchromatic tumor nuclei, is a hallmark of this aggressive brain tumor. Additionally, the recognition of hyperplastic glomeruloid microvascular proliferation aligns with the typical histological presentation of glioblastoma. Furthermore, the alignment with WHO CNS Grade 4 and the sensitivity to $O^6$-alkylating chemotherapy, particularly in cases with MGMT promoter hypermethylation, supports the diagnosis of glioblastoma. This comprehensive analysis confirms the diagnosis and highlights the sensitivity to chemotherapy in this condition.


## vq metalliger bfloat 16

In [1]:
from PIL import Image
import torch
from metalliger.inference import InferenceEngine

engine = InferenceEngine(
    model_id="Qwen3VL-2B-Medmaxv11",
    max_seq_len=4096,
    dtype=torch.bfloat16,       # ← Half the memory, what model was fine-tuned in
    use_compile=False
)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Initializing M4 Pro Engine for Qwen3VL-2B-Medmaxv11 (dtype=torch.bfloat16)...


Loading weights: 100%|█████████████████████| 626/626 [00:03<00:00, 156.80it/s]
MetalLiger Flash Attention: mps_flash_attn reports not available (C++ extension may not be built). Skipping.


- Injecting MetalLiger Fused Kernels (RMSNorm, SwiGLU)...
- Flash Attention: active (prefill speedup)
   RMSNorm fast-path: 57 layers optimized (fused F.rms_norm)
   SwiGLU fast-path: 28 MLP layers optimized for inference (torch.bfloat16)
- Running in stable eager mode with kernel fusions.
 Engine ready in 5.96s


In [5]:


def load_direct_image(path: str, max_edge: int = 512) -> Image.Image:
    with Image.open(path) as raw:
        img = raw.convert("RGB")
        width, height = img.size
        if max(width, height) > max_edge:
            scale = max_edge / max(width, height)
            new_w = int(width * scale)
            new_h = int(height * scale)
            img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        import io
        import base64
        buffered = io.BytesIO()
        img.save(buffered, format="JPEG")
        return base64.b64encode(buffered.getvalue()).decode("utf-8")#img


# 3. Prompt & Image
prompt = """You are a expert medical AI given a Histopathology Case (Glioblastoma / Pseudopalisading Necrosis)


1. Detection of central pseudopalisading necrosis (serpiginous necrotic core flanked by crowded hyperchromatic tumor nuclei).
2. Recognition of hyperplastic glomeruloid microvascular proliferation.
3. Alignment with WHO CNS Grade 4 / Glioblastoma and associated $O^6$-alkylating chemotherapy sensitivity (MGMT promoter hypermethylation).
"""

image_path = "GBM_pseudopalisading_necrosis.jpg"
img = load_direct_image(image_path)
# 4. Messages format
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }
]

# 5. Generate
response = engine.generate(
    messages=messages,
    images=[img],
    max_new_tokens=1048,
    temperature=0.0,
    verbose=True
)
print("\n--- Output ---")
print(response)

Prefilling prompt...
   Prefill done in 0.426s (732.9 tokens/s, 312 tokens)
⚡ Decoding...
   Decode done in 6.797s (47.7 tokens/s, 324 tokens)

--- Output ---
</think>

Based on the provided histopathological image, the following observations can be made:

1. **Detection of Central Pseudopalisading Necrosis**: The image shows a serpiginous necrotic core surrounded by crowded hyperchromatic tumor nuclei. This is characteristic of central pseudopalisading necrosis, a hallmark feature of glioblastoma.

2. **Recognition of Hyperplastic Glomeruloid Microvascular Proliferation**: The image also displays hyperplastic glomeruloid microvascular proliferation, which is another key feature of glioblastoma. This proliferation is often associated with the tumor's aggressive nature and poor prognosis.

3. **Alignment with WHO CNS Grade 4 / Glioblastoma**: The presence of central pseudopalisading necrosis and hyperplastic glomeruloid microvascular proliferation aligns with the diagnosis of WHO CNS Gr

## vq transformer float 32

In [1]:
import time
import torch
import transformers
from transformers import AutoConfig, AutoProcessor, Qwen3VLForConditionalGeneration
from PIL import Image

BASE_MODEL_ID = "Qwen3VL-2B-Medmaxv11"


model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    dtype=torch.float32,
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID, 
    trust_remote_code=True,
)
model.eval()


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████| 626/626 [00:00<00:00, 49025.04it/s]


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [2]:
import re
import time


def load_direct_image(path: str, max_edge: int = 512) -> Image.Image:
    with Image.open(path) as raw:
        img = raw.convert("RGB")
        width, height = img.size
        if max(width, height) > max_edge:
            scale = max_edge / max(width, height)
            new_w = int(width * scale)
            new_h = int(height * scale)
            img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        import io
        import base64
        buffered = io.BytesIO()
        img.save(buffered, format="JPEG")
        return base64.b64encode(buffered.getvalue()).decode("utf-8")#img


# 5. Formulate the Query

prompt = """You are given a Histopathology Case (Glioblastoma / Pseudopalisading Necrosis)


1. Detection of central pseudopalisading necrosis (serpiginous necrotic core flanked by crowded hyperchromatic tumor nuclei).

2. Recognition of hyperplastic glomeruloid microvascular proliferation.

3. Alignment with WHO CNS Grade 4 / Glioblastoma and associated $O^6$-alkylating chemotherapy sensitivity (MGMT promoter hypermethylation).
"""

img = load_direct_image("GBM_pseudopalisading_necrosis.jpg")

messages = [
    {
        "role": "system",
        "content": "You are an expert medical AI."
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }
]


with torch.inference_mode():
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=text,
        images=[img],
        return_tensors="pt",
    ).to("mps")

    # --- Start Benchmarking ---
    torch.mps.synchronize()  # Wait for all previous GPU tasks to finish
    start_time = time.perf_counter()

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=1024,      # tight control (prevents drift)
        do_sample=False,        # deterministic output
 
    )
    
    torch.mps.synchronize()  # Wait for generation to completely finish
    end_time = time.perf_counter()
    # --- End Benchmarking ---

    # Isolate only the newly generated tokens
    new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]

    output_text = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0]

    # Calculate Speed Metrics
    num_generated_tokens = new_tokens.shape[1]
    generation_time = end_time - start_time
    tokens_per_second = num_generated_tokens / generation_time


# Print Results
print("=== Output ===")
print(output_text.strip())
print("\n=== Performance Metrics ===")
print(f"Total New Tokens:  {num_generated_tokens}")
print(f"Generation Time:   {generation_time:.4f} seconds")
print(f"Token Speed:       {tokens_per_second:.2f} tokens/sec")

=== Output ===
</think>

The histopathological features observed in this image are characteristic of glioblastoma, specifically showing central pseudopalisading necrosis and hyperplastic glomeruloid microvascular proliferation. These features align with the WHO CNS Grade 4 classification, which is associated with glioblastoma. Additionally, the presence of $O^6$-alkylating chemotherapy sensitivity is indicated by the recognition of MGMT promoter hypermethylation, which is a common feature in glioblastoma. This sensitivity to chemotherapy is a critical factor in the treatment planning for patients with glioblastoma.

=== Performance Metrics ===
Total New Tokens:  122
Generation Time:   4.7623 seconds
Token Speed:       25.62 tokens/sec


## vq transformer bfloat 16

In [1]:
import os
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"

import time
import torch
import transformers
from transformers import AutoConfig, AutoProcessor, Qwen3VLForConditionalGeneration
from PIL import Image

BASE_MODEL_ID = "Qwen3VL-2B-Medmaxv11"


model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID, 
    trust_remote_code=True,
)
model.eval()


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 626/626 [00:02<00:00, 252.51it/s]


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [2]:
import re
import time


def load_direct_image(path: str, max_edge: int = 512) -> Image.Image:
    with Image.open(path) as raw:
        img = raw.convert("RGB")
        width, height = img.size
        if max(width, height) > max_edge:
            scale = max_edge / max(width, height)
            new_w = int(width * scale)
            new_h = int(height * scale)
            img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        import io
        import base64
        buffered = io.BytesIO()
        img.save(buffered, format="JPEG")
        return base64.b64encode(buffered.getvalue()).decode("utf-8")#img


# 5. Formulate the Query

prompt = """You are given a Histopathology Case (Glioblastoma / Pseudopalisading Necrosis)


1. Detection of central pseudopalisading necrosis (serpiginous necrotic core flanked by crowded hyperchromatic tumor nuclei).

2. Recognition of hyperplastic glomeruloid microvascular proliferation.

3. Alignment with WHO CNS Grade 4 / Glioblastoma and associated $O^6$-alkylating chemotherapy sensitivity (MGMT promoter hypermethylation).
"""

img = load_direct_image("GBM_pseudopalisading_necrosis.jpg")

messages = [
    {
        "role": "system",
        "content": "You are an expert medical AI."
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }
]


with torch.inference_mode():
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=text,
        images=[img],
        return_tensors="pt",
    ).to("mps")

    # --- Start Benchmarking ---
    torch.mps.synchronize()  # Wait for all previous GPU tasks to finish
    start_time = time.perf_counter()

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=1024,      # tight control (prevents drift)
        do_sample=False,        # deterministic output
 
    )
    
    torch.mps.synchronize()  # Wait for generation to completely finish
    end_time = time.perf_counter()
    # --- End Benchmarking ---

    # Isolate only the newly generated tokens
    new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]

    output_text = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0]

    # Calculate Speed Metrics
    num_generated_tokens = new_tokens.shape[1]
    generation_time = end_time - start_time
    tokens_per_second = num_generated_tokens / generation_time


# Print Results
print("=== Output ===")
print(output_text.strip())
print("\n=== Performance Metrics ===")
print(f"Total New Tokens:  {num_generated_tokens}")
print(f"Generation Time:   {generation_time:.4f} seconds")
print(f"Token Speed:       {tokens_per_second:.2f} tokens/sec")

=== Output ===
</think>

The histopathological features observed in this image are characteristic of glioblastoma, a highly aggressive brain tumor. The central pseudopalisading necrosis, which appears as a serpiginous necrotic core surrounded by crowded hyperchromatic tumor nuclei, is a hallmark of this condition. Additionally, the presence of hyperplastic glomeruloid microvascular proliferation is also noted, which is another typical feature of glioblastoma. These features align with the WHO CNS Grade 4 classification, indicating a glioblastoma. Furthermore, the sensitivity to $O^6$-alkylating chemotherapy, such as methotrexate, is associated with MGMT promoter hypermethylation, which is a common genetic alteration in glioblastoma. Therefore, this histopathological case is consistent with the diagnosis of glioblastoma and its associated treatment sensitivity.

=== Performance Metrics ===
Total New Tokens:  180
Generation Time:   4.6339 seconds
Token Speed:       38.84 tokens/sec
